# ID6004W Industrial Vision - Assignment 2
### Lectures 03-06: Edges, Lines, Corners, and Scale

| | |
|---|---|
| **Released** | Oct 1st 2026 |
| **Due** | Oct 21st 2026 (5.00 PM IST) |
| **Total** | 100 marks (Theory 65 + Programming 35; will be scaled down appropriately later) |

## Instructions

**Working in Colab.** Open this notebook in Google Colab and immediately use *File → Save a copy in Drive*; work only in your copy. Type your answers in the cells marked **Your answer**, directly below each question (double-click a cell to edit it). Mathematics can be typed in LaTeX between `$...$` or `$$...$$`. Photographs of hand-written working are **not** accepted in place of typed answers.

- **The writing must be your own**, for both the theory answers and the code.
- Cells marked **Optional check** are not graded. Use them to verify a hand calculation numerically, but the answer you submit must be derived by hand with the steps shown.
- Part B is done in this notebook. Run every cell from top to bottom (*Runtime → Restart and run all*) before you submit, so that all outputs and figures are visible in the file.
- **Submission.** When you are done, use *File → Download → Download .ipynb*, rename the file to `<ROLLNO>_assignment2.ipynb`, and upload it here: [SUBMISSION LINK](https://forms.gle/FqCHkx8jMZyQQ4fe7). For queries contact Rishabh Shukla (da26e011@smail.iitm.ac.in). The form link will be disabled after the deadline. Follow the naming convention strictly. If you used your own image in Part B, also say which one in the answer cells.
- **Late policy:** 10 marks will be deducted for each day the assignment is submitted late after the due date.

## Student details

- **Name:**
- **Roll number:**

In [ ]:
# Setup: run this cell once. It downloads the data for this assignment into the Colab session.
# Colab deletes session files when the runtime restarts; just run the cell again.
BASE_URL = "https://seafile.unistra.fr/f/421d01267ea6404a8575/?dl=1"   # set by the instructor, do not change
DATA = "assignment2_data"

import os
if not os.path.isdir(DATA):
    !wget --content-disposition -q -O {DATA}.zip "{BASE_URL}/{DATA}.zip"
    !unzip -q -o {DATA}.zip && rm -f {DATA}.zip
print(DATA, "->", sorted(os.listdir(DATA)))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

---
# Part A - Theory  [65 marks]

---
## Question 1. The Canny edge detector  [10 marks]

**(a) Non-maximum suppression by hand.** **[5]**

At the pixel $a=(x,y)$ the gradient magnitude is $\lVert\nabla G\rVert_a = 42.0$ and the gradient orientation is $\theta=20^{\circ}$. The magnitudes at $a$ and its eight neighbours are

| | $x-1$ | $x$ | $x+1$ |
|:-:|:-:|:-:|:-:|
| $y-1$ | 18.0 | 27.0 | 36.0 |
| $y$ | 39.0 | **42.0** | 40.0 |
| $y+1$ | 12.0 | 25.0 | 33.0 |

1. Round $\theta$ to the nearest of the four directions $0^{\circ},45^{\circ},90^{\circ},135^{\circ}$, state which two neighbours are then compared with $a$, and say whether $a$ is retained.
2. Now use the interpolated form of the test. Compute the magnitude at the forward point $a+(\cos\theta,\sin\theta)$ and at the backward point $a-(\cos\theta,\sin\theta)$ by bilinear interpolation of the four surrounding pixels, and say whether $a$ is retained.

**Your answer 1(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [ ]:
# Optional check for 1(a). Rows are y-1, y, y+1 and columns are x-1, x, x+1, so a sits at [1, 1].
mag = np.array([[18.0, 27.0, 36.0],
                [39.0, 42.0, 40.0],
                [12.0, 25.0, 33.0]])
theta = np.deg2rad(20)

# TODO (optional): write bilinear(mag, x, y) and evaluate it at the forward and backward points.

**(b) Hysteresis by hand.** **[5]**

After non-maximum suppression, a $1\times9$ strip along an edge chain has magnitudes

$$[\,80,\;\;55,\;\;22,\;\;18,\;\;14,\;\;19,\;\;26,\;\;60,\;\;9\,]$$

with a high threshold $T_2=50$ and a low threshold $T_1=25$. List the indices that are retained, and state which of them are retained only because of the linking rule. Then state, in one sentence, the assumption about edges in real images that makes hysteresis a reasonable rule.

**Your answer 1(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
## Question 2. The Hough transform  [15 marks]

**(a) The point-line duality.** **[5]**

Take the three collinear points $(1,3)$, $(3,5)$, $(5,7)$.

1. Find the pair $(\rho,\theta)$ with $\theta\in[0,\pi)$ that represents the line through them, and verify by substitution that all three points vote for that bin.
2. Add a fourth point $(2,7)$, which is not on that line. Its sinusoid crosses each of the other three sinusoids. Explain why these crossings do not produce a competing peak, and state the general rule relating the number of votes in a bin to the structure in the image.

**Your answer 2(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(b) Choosing the accumulator resolution.** **[5]**

Consider a $512\times512$ image and a straight edge of length $L$ pixels.

1. Taking the origin at the centre of the image, state the range of $\rho$ and the number of accumulator cells for $\Delta\rho=1$ px and $\Delta\theta=1^{\circ}$.
2. Explain why making $\Delta\theta$ coarser can *lower* the peak vote count, and state what goes wrong when $\Delta\theta$ is made very fine.

**Your answer 2(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(c) The limits of the accumulator.** **[5]**

1. Explain in one or two sentences why deleting a random half of a line's edge pixels barely affects its detection.
2. A photograph of a doorway contains two collinear vertical segments separated by a wide horizontal lintel. The standard Hough transform reports a single line. Explain why, and state what information would have to be added to the accumulator to separate the two segments.

**Your answer 2(c).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
## Question 3. The structure matrix  [10 marks]

Recall from Lecture 04 that the change produced by a shift $(\Delta x,\Delta y)$ of the window $W$ is

$$
C(\Delta x,\Delta y)\;\approx\;
\begin{bmatrix}\Delta x & \Delta y\end{bmatrix}\,R\,
\begin{bmatrix}\Delta x \\ \Delta y\end{bmatrix},
\qquad
R=\sum_{W} w\begin{bmatrix} I_x^{2} & I_xI_y \\ I_xI_y & I_y^{2}\end{bmatrix},
$$

where $w$ is the window weight at each pixel.

**(a) Three patches by hand.** **[5]**

Three $3\times3$ windows are described by the gradient values $(I_x,I_y)$ at their nine pixels. Take an unweighted window, $w\equiv1$.

| Patch | Gradients |
|:-:|:--|
| $P$ | $(3,4)$ five times, $(0,0)$ four times |
| $Q$ | $(3,4)$ three times, $(-4,3)$ three times, $(0,0)$ three times |
| $S$ | $(1,0),(0,1),(-1,0),(0,-1)$, and $(0,0)$ five times |

The same three windows are drawn by the cell below. Each cell is one pixel, each arrow is that pixel's gradient $(I_x,I_y)$, and a grey dot marks a pixel whose gradient is zero. The arrows of all three patches are drawn to one common scale, and the $x$-axis points right and the $y$-axis points down, as usual for image coordinates.

In [ ]:
# Run to draw the three patches of Question 3(a).
patches = {   # (column, row) -> (Ix, Iy)
    "Patch P": {(0, 2): (3, 4), (1, 1): (3, 4), (2, 0): (3, 4), (1, 2): (3, 4), (2, 1): (3, 4)},
    "Patch Q": {(0, 0): (3, 4), (1, 0): (3, 4), (2, 0): (3, 4),
                (0, 2): (-4, 3), (1, 2): (-4, 3), (2, 2): (-4, 3)},
    "Patch S": {(0, 0): (1, 0), (2, 0): (0, 1), (0, 2): (0, -1), (2, 2): (-1, 0)},
}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (name, g) in zip(axes, patches.items()):
    for col in range(3):
        for row in range(3):
            gx, gy = g.get((col, row), (0, 0))
            if gx == 0 and gy == 0:
                ax.plot(col + 0.5, row + 0.5, "o", color="0.6", ms=3)
            else:
                ax.quiver(col + 0.5, row + 0.5, gx, gy, angles="xy", scale_units="xy", scale=7, pivot="middle",
                          color="navy", width=0.012)
    ax.set_xlim(0, 3); ax.set_ylim(3, 0)          # y axis points down
    ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.grid(True, color="0.8")
    ax.tick_params(labelbottom=False, labelleft=False, length=0)
    ax.set_aspect("equal"); ax.set_title(name)
plt.show()

*Note that $R$ is a sum over the pixels of the window, so it depends only on which gradient values occur and how often, and not on where in the window they sit. The arrangements drawn above are therefore one plausible choice out of many: in $P$ the five equal gradients lie along a single edge running through the window, in $Q$ the window contains two edge fragments whose gradients are perpendicular, and in $S$ the four gradients point in unrelated directions and are five times shorter than those of $P$ and $Q$.*

For each patch, write down the three sums $\sum I_x^{2}$, $\sum I_y^{2}$, $\sum I_xI_y$ and hence the matrix $R$; compute $\lambda_1\ge\lambda_2$ and the eigenvectors; and classify the patch as corner, edge or flat. For the edge patch, state the orientation of the edge and check that it agrees with your eigenvector. *(Keep these three matrices; Question 4 reuses them.)*

**Your answer 3(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [ ]:
# Optional check for 3(a) and 4(a): the gradient lists of the three patches.
grads = {
    "P": [(3, 4)] * 5 + [(0, 0)] * 4,
    "Q": [(3, 4)] * 3 + [(-4, 3)] * 3 + [(0, 0)] * 3,
    "S": [(1, 0), (0, 1), (-1, 0), (0, -1)] + [(0, 0)] * 5,
}

# TODO (optional): build R for each patch, then use np.linalg.eigh to check your eigenvalues and
# eigenvectors, and compute det R, tr R and M = det R - 0.04 tr^2 R for Question 4(a).

**(b) The two Gaussians.** **[5]**

Lecture 04 introduced two distinct Gaussians: $\sigma_s$, which smooths the image before differentiation, and $\sigma_w$, which weights the window.

1. State the purpose of each, and explain why they are independent parameters rather than one.
2. Let the window shrink to a single pixel, so that $R$ is built from one gradient $(I_x,I_y)$ alone. Compute $\det R$ and $\operatorname{tr} R$ in this case, and hence the Harris response $M=\det R-\kappa\,\operatorname{tr}^2R$. State what the detector now reports everywhere in every image, and what this tells you about where cornerness resides.

**Your answer 3(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
## Question 4. The Harris response and sub-pixel localisation  [20 marks]

**(a) The response on the three patches.** **[5]**

Using $R_P,R_Q,R_S$ from Question 3(a), compute $M=\det R-\kappa\operatorname{tr}^{2}R$ for $\kappa=0.04$ and tabulate $\det R$, $\operatorname{tr} R$ and $M$. The flat patch gives $M>0$. State precisely what the sign of $M$ does and does not tell you, and quote the ratio $M_S/M_Q$ in support of your answer.

**Your answer 4(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(b) The meaning of $\kappa$.** **[5]**

Let $r=\lambda_1/\lambda_2\ge1$ measure the elongation of the patch.

1. Starting from $M=\lambda_1\lambda_2-\kappa(\lambda_1+\lambda_2)^{2}$, divide by $\lambda_2^{2}$ and show that $M>0$ is equivalent to $\kappa\,r^{2}-(1-2\kappa)\,r+\kappa<0$. Solve this quadratic for $\kappa=0.04$ and for $\kappa=0.06$, and state the largest $r$ accepted in each case. Conclude that $\kappa$ acts as a threshold on the elongation.
2. SIFT rejects a keypoint when its curvature ratio exceeds $r_{\max}=10$ (see the Lecture 07 slides). Compare this with your answer to (1), and state in one or two sentences what design decision $\kappa$ and $r_{\max}$ are two ways of expressing.

**Your answer 4(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(c) Invariance.** **[5]**

1. For $I\to aI+b$ with $a>0$, derive how the entries of $R$, the eigenvalues $\lambda_i$ and the response $M$ transform. State the consequence for a fixed threshold $T$ and give the standard remedy.
2. Rotate the image. The gradient at a corresponding point is the original gradient rotated by the same angle. Explain why the eigenvalues of $R$, and therefore $M$, are unchanged, and state the one assumption about the window that this argument requires. Do typical implementations satisfy it?
3. Explain in words why the same argument fails for a change of scale, in which the image is magnified by a factor $c$ while the window remains a fixed number of pixels.

**Your answer 4(c).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(d) Sub-pixel refinement by hand.** **[5]**

Near a detected maximum at $(x_c,y_c)$ the Harris response map takes the values

| | $x-1$ | $x$ | $x+1$ |
|:-:|:-:|:-:|:-:|
| $y-1$ | 0.20 | 0.55 | 0.45 |
| $y$ | 0.32 | **1.00** | 0.86 |
| $y+1$ | 0.18 | 0.60 | 0.52 |

1. Using the kernels of Lecture 05, namely $f_x:\tfrac12[1\ 0\ {-}1]$, $f_{xx}:[1\ {-}2\ 1]$ and $f_{xy}:\tfrac14\begin{bmatrix}1&0&-1\\0&0&0\\-1&0&1\end{bmatrix}$, compute $\nabla f$ and $H$, and solve $\Delta=-H^{-1}\nabla f$.
2. Verify from the signs of $\operatorname{tr} H$ and $\det H$ that the refined point is a maximum and not a saddle.

**Your answer 4(d).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [ ]:
# Optional check for 4(d). Rows are y-1, y, y+1 and columns are x-1, x, x+1.
f = np.array([[0.20, 0.55, 0.45],
              [0.32, 1.00, 0.86],
              [0.18, 0.60, 0.52]])

# TODO (optional): compute fx, fy, fxx, fyy, fxy at the centre, then Delta = -np.linalg.solve(H, grad).

---
## Question 5. Scale space and scale normalisation  [10 marks]

**(a) Why the factor $\sigma^{2}$.** **[5]**

1. For an ideal step $I(x)=u(x-x_0)$, derive the LoG response $R_\sigma(x)=u\ast \tfrac{d^{2}G_\sigma}{dx^{2}}$, show that its extrema lie at $x=x_0\pm\sigma$, and evaluate $R_\sigma(x_0\pm\sigma)$.
2. Hence explain why the unnormalised LoG response cannot be compared across scales, and why multiplying by $\sigma^{2}$ is the exact correction rather than merely a convenient one.

**Your answer 5(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(b) What a detected $\sigma$ means.** **[5]**

1. Using the zero set of the 2-D LoG, derive $r=\sqrt2\,\sigma$ for a binary disc of radius $r$, and state the geometric picture in one sentence.

**Your answer 5(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
# Part B - Programming  [35 marks]

## Question 6. Canny edge detector implementation  [35 marks]

Implement the four stages yourself: derivative-of-Gaussian gradients, magnitude and orientation, non-maximum suppression with interpolated neighbours, and hysteresis with 8-connected linking. Show four panels: $\lVert\nabla G\rVert$, the map after thresholding at $T_1$ only, the map after non-maximum suppression, and the final result. Compare your output with `cv2.Canny` on the same image. Report the fraction of your edge pixels lying within $1$ px of a `cv2.Canny` edge pixel, and account for at least two systematic differences. Note that `cv2.Canny` uses a fixed $3\times3$ Sobel operator and by default the $L_1$ magnitude $|G_x|+|G_y|$; verify the effect of the `L2gradient` flag.

**Data.** The setup cell downloaded `assignment2_data/building.jpg` (a photograph with long straight edges) and `assignment2_data/scene.png` (a synthetic image with ideal step edges and exact corners, useful for debugging). Develop on `scene.png`, then report on `building.jpg`. You may additionally use an image of your own (upload it with the file browser on the left).

**Rules.** You may use NumPy, SciPy and OpenCV for image I/O and for applying a kernel that *you* have built (for example `scipy.ndimage.convolve1d` or `cv2.sepFilter2D`). The four stages themselves must be your own code: do not use `cv2.Canny` (except for the comparison), `cv2.Sobel`, `cv2.GaussianBlur`, `skimage.feature.canny`, or a library connected-components routine for the hysteresis. Vectorised NumPy is expected where it is natural; a loop over the edge pixels in the linking step is fine.

In [ ]:
from scipy import ndimage

img_bgr = cv2.imread(f"{DATA}/building.jpg")          # change the file name to try another image
img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float64)
plt.figure(figsize=(6, 5)); plt.imshow(img, cmap="gray"); plt.axis("off"); plt.show()
print(img.shape, img.min(), img.max())

### Stage 1 and 2: derivative-of-Gaussian gradients, magnitude and orientation

In [ ]:
def gaussian_kernels(sigma):
    # Return the sampled 1-D Gaussian g and its derivative dg on x = -r..r with r = ceil(3 sigma).
    # Normalise g to sum to 1, and dg so that it returns the slope of a unit ramp.
    # TODO
    g, dg = None, None
    return g, dg


def gradients(img, sigma):
    # Return Ix, Iy: the derivative of the Gaussian-smoothed image along x (columns) and y (rows),
    # computed with separable 1-D passes of the kernels above.
    # TODO
    Ix, Iy = None, None
    return Ix, Iy


def magnitude_orientation(Ix, Iy):
    # Return the L2 gradient magnitude and the orientation theta (radians).
    # TODO
    mag, theta = None, None
    return mag, theta

### Stage 3: non-maximum suppression with interpolated neighbours

In [ ]:
def non_max_suppression(mag, Ix, Iy):
    # Keep a pixel only if its magnitude is >= the magnitudes at a + u and a - u, where
    # u = (cos theta, sin theta) is the unit gradient direction and the two values are read by
    # bilinear interpolation of the four surrounding pixels. Return the suppressed magnitude map
    # (zero where the pixel is suppressed). Treat the one-pixel border as suppressed.
    # TODO
    nms = None
    return nms

### Stage 4: hysteresis with 8-connected linking

In [ ]:
def hysteresis(nms, T1, T2):
    # T1 < T2. Pixels with nms >= T2 are strong edges. A pixel with T1 <= nms < T2 is kept only if it
    # is 8-connected, through a chain of such pixels, to a strong edge. Return a boolean edge map.
    # TODO
    edges = None
    return edges


def my_canny(img, sigma, T1, T2):
    # Given: runs the four stages and returns every intermediate map for the figures.
    Ix, Iy = gradients(img, sigma)
    mag, theta = magnitude_orientation(Ix, Iy)
    nms = non_max_suppression(mag, Ix, Iy)
    edges = hysteresis(nms, T1, T2)
    return dict(Ix=Ix, Iy=Iy, mag=mag, theta=theta, nms=nms, edges=edges)

### Results: the four panels

Choose $\sigma$, $T_1$ and $T_2$ for your image and justify the choice in the answer cell at the end.

In [ ]:
SIGMA, T1, T2 = 1.4, None, None      # TODO: choose the thresholds (in units of your gradient magnitude)

out = my_canny(img, SIGMA, T1, T2)
panels = [("gradient magnitude", out["mag"]),
          ("magnitude >= T1 only", out["mag"] >= T1),
          ("after non-maximum suppression", out["nms"] > 0),
          ("final (hysteresis)", out["edges"])]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (title, im) in zip(axes.ravel(), panels):
    ax.imshow(im, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()
print("edge pixels:", int(out["edges"].sum()))

### Comparison with `cv2.Canny`

`cv2.Canny` works on the unsmoothed 8-bit image with a $3\times3$ Sobel operator, so its thresholds are on a different scale from yours; choose `lo`, `hi` so that the two detectors report a comparable number of edge pixels. Smooth the input with your own Gaussian first if you want a fairer comparison, and say whether you did.

In [ ]:
def fraction_within_1px(mine, ref):
    # Fraction of the True pixels of `mine` that lie within 1 px (8-neighbourhood) of a True pixel of `ref`.
    # TODO
    return None


lo, hi = None, None                  # TODO: cv2.Canny thresholds
img_u8 = img.astype(np.uint8)
cv_l1 = cv2.Canny(img_u8, lo, hi, L2gradient=False) > 0
cv_l2 = cv2.Canny(img_u8, lo, hi, L2gradient=True) > 0

for name, ref in (("cv2.Canny L1", cv_l1), ("cv2.Canny L2", cv_l2)):
    print(f"{name}: {int(ref.sum())} edge pixels; fraction of mine within 1 px: "
          f"{fraction_within_1px(out['edges'], ref)}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (title, im) in zip(axes, (("mine", out["edges"]), ("cv2.Canny, L1", cv_l1), ("cv2.Canny, L2gradient=True", cv_l2))):
    ax.imshow(im, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Overlay: yours in red, cv2.Canny (L2) in green, both in yellow.
overlay = np.zeros(img.shape + (3,))
overlay[..., 0] = out["edges"]
overlay[..., 1] = cv_l2
plt.figure(figsize=(10, 8)); plt.imshow(overlay); plt.axis("off"); plt.show()

### Report for Question 6

Answer in the cell below:

1. The image(s) you used and your choice of $\sigma$, $T_1$, $T_2$ (and `lo`, `hi` for `cv2.Canny`), with a short justification.
2. The fraction of your edge pixels lying within 1 px of a `cv2.Canny` edge pixel, for both settings of `L2gradient`.
3. At least two systematic differences between your output and `cv2.Canny`, each explained by a specific difference between the two implementations, with a pointer to where it is visible in the figures.
4. What changes when `L2gradient=True`, and why.

**Your answer 6.**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
*End of Assignment 2.*